# Knowledge Graph Extraction — Pipeline 4 Passes (v7 — Robust)

**Fixes vs v6:**
- ✅ `batch_size` → `max_batch_size` (keyword arg typo)
- ✅ Retry logic: exponential backoff (20s → 40s → 80s) instead of flat 15s × 2
- ✅ JSON repair with `json-repair` for malformed LLM output (fixes `[JSON ERROR]`)
- ✅ `call_llm_with_doc` now also uses exponential backoff (Pass 2 timeouts)
- ✅ Max chunk size reduced from 3000 → 2500 chars to reduce Pass 2 payload
- ✅ Pass 2 retry: on timeout, chunk is re-split and retried at half size
- ✅ All retry logic centralised in one `call_llm_retry()` function


## Bloc 1 — Imports & Configuration

In [ ]:
import os
import json
import time
import re
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import ChatNVIDIA

# json-repair: pip install json-repair
try:
    from json_repair import repair_json
    HAS_JSON_REPAIR = True
except ImportError:
    HAS_JSON_REPAIR = False
    print("WARNING: json-repair not installed. Run: pip install json-repair")

load_dotenv()

print("Imports OK")
print(f"NVIDIA_API_KEY : {'OK' if os.getenv('NVIDIA_API_KEY') else 'MISSING'}")
print(f"NEO4J_URI      : {os.getenv('NEO4J_URI', 'Not defined')}")
print(f"json-repair    : {'OK' if HAS_JSON_REPAIR else 'NOT INSTALLED — JSON errors won\'t be auto-repaired'}")


## Bloc 2 — LLM Initialization

In [ ]:
llm = ChatNVIDIA(
    model="mistralai/mistral-large-3-675b-instruct-2512",
    api_key=os.getenv("NVIDIA_API_KEY"),
    temperature=0,
    max_completion_tokens=32000,
)
print("LLM ready")


## Bloc 3 — Graph Schema

In [ ]:
SCHEMA = """
NODES (entity: fields):
  Company         : name, activity, certifications
  Supplier        : name
  Agreement       : name, description, type, startdate, enddate
  Document        : reference, title, version, date, owner
  Clients         : name, sector
  Clause          : name, description
  governance_body : name, acronyme, role
  Claim           : claim_type, value, unit, scope, source_doc, source_doc_id, quote
  Roles           : title
  Asset           : type, description, classification
  Algorithm       : name, use
  Protocol        : name, use, source
  Risk            : description, level
  Framework       : name, type, use
  Control         : name, requirement, source, Cycle
  Technology      : name, use, source

RELATIONSHIPS (source -[REL]-> target):
  Agreement       -[for]->              Supplier
  Company         -[Use]->              Agreement
  Supplier        -[HAS]->              Company
  Company         -[HAS]->              Document
  Company         -[HAS]->              Clients
  Company         -[HAS]->              governance_body
  Company         -[has_requirement]->  Clause
  Supplier        -[apply_to]->         Clause
  governance_body -[supervise]->        Roles
  governance_body -[approves]->         Claim
  Claim           -[Concerns]->         Asset
  Claim           -[Asserted_by]->      Roles
  Roles           -[operates]->         Technology
  Roles           -[EXECUTES]->         Control
  Algorithm       -[protects]->         Asset
  Protocol        -[used_by]->          Algorithm
  Protocol        -[mitigates]->        Risk
  Risk            -[targeting]->        Asset
  Framework       -[required_by]->      Control
  Control         -[IMPLEMENTED_BY]->   Technology
  Technology      -[Has_access_to]->    Asset
"""

NODE_LABELS = [
    "Company", "Supplier", "Agreement", "Document", "Clients",
    "Clause", "governance_body", "Claim", "Roles", "Asset",
    "Algorithm", "Protocol", "Risk", "Framework", "Control", "Technology"
]

print(f"Schema defined: {len(NODE_LABELS)} node types, 21 relationship types")


## Bloc 4 — Prompts for the 4 Passes

In [ ]:
# -----------------------------------------------------------------
# PASS 1 — Which entity types are present in this chunk?
# -----------------------------------------------------------------
PROMPT_PASS1 = f"""
You are a knowledge-graph analyst. Your task is to SCAN a document chunk
and identify which entity types from the schema are mentioned.

Valid schema node labels (these are the ONLY valid values to return):
{', '.join(NODE_LABELS)}

IMPORTANT — label definitions to avoid confusion:
- 'Cycle' is NOT a valid label. It is a field inside Control nodes, not a node type.
- 'Algorithm' = cryptographic or hashing algorithm (AES, SHA, RSA, ECC...)
- 'Protocol' = communication or security protocol (TLS, IPsec, CVSS, break-glass...)
- 'Technology' = software tool or platform (Azure, SIEM, Fortinet, ServiceNow...)
- 'Supplier' = external vendor, contractor, service provider, or subcontractor
- 'Clause' = contractual clause, policy rule, or stated obligation
- 'Claim' = measurable assertion with a value (RTO, RPO, KPI, rate, threshold...)

Rules:
- Be INCLUSIVE: if a concept is mentioned even implicitly, include its label.
- Return ONLY a JSON array of valid label strings.
- No explanation, no markdown, no extra text.

Output format example:
["Company", "Document", "Roles", "Control", "Supplier", "Clause"]
"""

# -----------------------------------------------------------------
# PASS 2 — Extract entities with their fields (targeted labels)
# -----------------------------------------------------------------
def build_prompt_pass2(detected_labels: list) -> str:
    schema_lines = []
    for line in SCHEMA.strip().split("\n"):
        for label in detected_labels:
            if line.strip().startswith(label):
                schema_lines.append(line)
                break
    focused_schema = "\n".join(schema_lines)

    key_fields = {
        'Company': 'name', 'Supplier': 'name', 'Agreement': 'name',
        'Document': 'reference', 'Clients': 'name', 'Clause': 'name',
        'governance_body': 'name', 'Claim': 'claim_type', 'Roles': 'title',
        'Asset': 'type', 'Algorithm': 'name', 'Protocol': 'name',
        'Risk': 'description', 'Framework': 'name', 'Control': 'name',
        'Technology': 'name'
    }
    key_reminder = "\n".join(
        f"  - {l}: '{key_fields[l]}' is MANDATORY and must not be empty"
        for l in detected_labels if l in key_fields
    )

    return f"""
You are a knowledge-graph extraction assistant.
Extract ALL entities of the following types from the document chunk.
Be exhaustive: extract every instance you can find.

Target entity types and their fields:
{focused_schema}

LABEL DISAMBIGUATION — avoid these common mistakes:
- 'Cycle' is NOT a node label. It is a FIELD inside Control (e.g. Control.Cycle = 'Annual').
  Never create an entity with label 'Cycle'.
- 'Algorithm' = cryptographic/hashing algorithm only (AES-256, SHA-512, RSA-4096, ECC...).
  Tools like Checkov, SonarQube, scripts → use label 'Technology' instead.
- 'Company' = named organization. NEVER use 'Unknown', 'Company', or a hallucinated name.
  If you cannot identify the company name from the text, do NOT extract a Company entity.
- 'Supplier' = external vendor, contractor, or subcontractor mentioned in the text.
- 'Clause' = an explicit rule, obligation, or contractual requirement stated in the document.
- 'Claim' = a measurable assertion with a value: RTO, RPO, KPI, rate, threshold, percentage.

MANDATORY KEY FIELDS — each entity MUST have a non-empty value for its key field:
{key_reminder}
If you cannot find a non-empty value for the key field, DO NOT extract that entity.

Rules:
- Use ONLY the node labels listed above.
- Extract every field you can find; omit fields that are not mentioned.
- String values only, no nested objects.
- NEVER invent or hallucinate values — only extract what is explicitly in the text.
- Keep property values in the ORIGINAL language of the document (translation happens in Pass 3).
- Return ONLY a valid JSON array, no markdown, no explanation.

Output format:
[
  {{"label": "<NodeLabel>", "properties": {{"field": "value"}}}},
  ...
]
"""

# -----------------------------------------------------------------
# PASS 3 — Translate all entity property values to English
# -----------------------------------------------------------------
def build_prompt_pass3_translate(entities: list) -> str:
    entities_json = json.dumps(entities, ensure_ascii=False, indent=2)
    return f"""
You are a translation assistant for a knowledge graph.
Below is a list of extracted entities with their properties.
The property VALUES may be in French or another language.

Your task:
- Translate ALL property VALUES to English — NO EXCEPTIONS.
- This includes job titles, role names, committee names, and governance body names.
  Example: 'Responsable GRC' -> 'GRC Manager', 'Comité Sécurité' -> 'Security Committee',
           'Direction Générale' -> 'General Management', 'Responsable PRA/PCA' -> 'BCP/DRP Manager'
- Keep property KEYS exactly as-is (do not translate field names).
- Keep entity LABELS exactly as-is (do not translate labels).
- Keep proper nouns that are product names or acronyms unchanged (e.g. Azure, SIEM, MFA, ISO 27001).
- If a value is already in English, leave it unchanged.
- Return the SAME JSON structure with translated values only.
- Return ONLY a valid JSON array, no markdown, no explanation.

Entities to translate:
{entities_json}

Output format (same structure, translated values):
[
  {{"label": "<NodeLabel>", "properties": {{"field": "translated value"}}}},
  ...
]
"""

# -----------------------------------------------------------------
# PASS 4 — Link relationships between translated entities
# -----------------------------------------------------------------
def build_prompt_pass4_relations(entities: list) -> str:
    key_fields = {
        'Company': 'name', 'Supplier': 'name', 'Agreement': 'name',
        'Document': 'reference', 'Clients': 'name', 'Clause': 'name',
        'governance_body': 'name', 'Claim': 'claim_type', 'Roles': 'title',
        'Asset': 'type', 'Algorithm': 'name', 'Protocol': 'name',
        'Risk': 'description', 'Framework': 'name', 'Control': 'name',
        'Technology': 'name'
    }
    entity_summary = []
    for e in entities:
        label = e['label']
        kf = key_fields.get(label)
        if kf and e['properties'].get(kf):
            entity_summary.append({"label": label, "key": {kf: e['properties'][kf]}})
    entity_summary_json = json.dumps(entity_summary, ensure_ascii=False)
    rel_schema = SCHEMA.split('RELATIONSHIPS')[1].strip()
    return f"""
You are a knowledge-graph relationship extractor.
You are given the EXACT list of entities that exist in the graph.
Your task: identify relationships between them that match the schema.

CRITICAL RULES:
- Do NOT invent or hallucinate entities. Only use entities from the list below.
- from_key and to_key MUST use the EXACT key field and value shown in the list.
- Do NOT read any source document — work exclusively from the entity list.
- Use ONLY relationship types from the schema.
- Return ONLY a valid JSON array, no markdown, no explanation.

Schema relationships:
{rel_schema}

Entity list (label + exact key field to use in from_key/to_key):
{entity_summary_json}

Output format:
[
  {{
    "from_label": "<NodeLabel>",
    "from_key":   {{"<key_field>": "<exact_value_from_list>"}},
    "rel_type":   "<REL_TYPE>",
    "to_label":   "<NodeLabel>",
    "to_key":     {{"<key_field>": "<exact_value_from_list>"}}
  }},
  ...
]
"""

print("Prompts for all 4 passes defined")


## Bloc 5 — Utility Functions (retry + JSON repair)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Core LLM call (no retry here)
# ─────────────────────────────────────────────────────────────────
def call_llm_raw(system_prompt: str, user_text: str) -> str:
    """Call the LLM and return raw content string."""
    messages = [("system", system_prompt), ("human", user_text)]
    raw = llm.invoke(messages).content.strip()
    # Strip markdown fences if present
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
        raw = raw.strip()
    return raw


# ─────────────────────────────────────────────────────────────────
# Centralised retry with EXPONENTIAL BACKOFF
# Fixes: flat 15s × 2 was too short and retried immediately into same timeout
# ─────────────────────────────────────────────────────────────────
def call_llm_retry(
    system_prompt: str,
    user_text: str,
    max_retries: int = 4,
    base_wait: int = 20,
) -> str:
    """
    Call LLM with exponential backoff retry.
    Waits: 20s, 40s, 80s, 160s between attempts (base_wait * 2^attempt).
    max_retries=4 gives up to ~5 min of retry window for transient NVIDIA load spikes.
    """
    last_exc = None
    for attempt in range(1, max_retries + 1):
        try:
            return call_llm_raw(system_prompt, user_text)
        except Exception as e:
            last_exc = e
            if attempt < max_retries:
                wait = base_wait * (2 ** (attempt - 1))  # 20, 40, 80, 160
                print(f"retry {attempt}/{max_retries - 1} in {wait}s...", end=" ", flush=True)
                time.sleep(wait)
    raise last_exc


def call_llm_with_doc(system_prompt: str, doc_text: str, max_retries: int = 4) -> str:
    """Pass 1 & 2: call LLM with document chunk as user input."""
    return call_llm_retry(
        system_prompt,
        f"Document chunk:\n\n{doc_text}",
        max_retries=max_retries,
    )


def call_llm_no_doc(system_prompt: str, max_retries: int = 4) -> str:
    """Pass 3 & 4: call LLM without any document."""
    return call_llm_retry(
        system_prompt,
        "Process the entities listed in the instructions above.",
        max_retries=max_retries,
    )


# ─────────────────────────────────────────────────────────────────
# JSON parsing with auto-repair
# Fixes: [JSON ERROR] Expecting ':' delimiter on malformed LLM output
# ─────────────────────────────────────────────────────────────────
def parse_json(raw: str, context: str = ""):
    """
    Parse JSON with fallback repair.
    1. Try standard json.loads
    2. Try json_repair.repair_json (handles truncated / malformed LLM output)
    3. Try extracting the first [...] or {...} block via regex
    """
    # 1. Standard parse
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        pass

    # 2. json-repair
    if HAS_JSON_REPAIR:
        try:
            repaired = repair_json(raw)
            parsed = json.loads(repaired)
            print(f"      [JSON REPAIRED] {context}", end=" ")
            return parsed
        except Exception:
            pass

    # 3. Regex extraction fallback
    for pattern in [r'(\[.*?\])', r'(\{.*?\})']:
        match = re.search(pattern, raw, re.DOTALL)
        if match:
            try:
                parsed = json.loads(match.group(1))
                print(f"      [JSON EXTRACTED] {context}", end=" ")
                return parsed
            except json.JSONDecodeError:
                pass

    print(f"      [JSON ERROR — unrecoverable] {context} : {raw[:200]}")
    return None


# ─────────────────────────────────────────────────────────────────
# Document splitting utilities
# ─────────────────────────────────────────────────────────────────
def split_by_document(text: str, marker: str = "=== DOCUMENT") -> list:
    chunks = []
    parts = text.split(marker)
    for part in parts[1:]:
        first_line = part.split("\n")[0].strip().rstrip("===").strip()
        doc_id = f"DOCUMENT {first_line}"
        chunks.append({"id": doc_id, "text": (marker + part).strip()})
    return chunks if chunks else [{"id": "FULL", "text": text}]


def split_long_chunk(chunk: dict, max_chars: int = 2500) -> list:
    """
    Split at paragraph boundaries. max_chars reduced to 2500 (was 3000)
    to stay well under the NVIDIA timeout threshold.
    """
    text = chunk["text"]
    doc_id = chunk["id"]

    if len(text) <= max_chars:
        return [chunk]

    paragraphs = text.split("\n\n")
    sub_chunks, current = [], ""

    for para in paragraphs:
        if len(current) + len(para) + 2 > max_chars and current:
            sub_chunks.append(current.strip())
            current = para
        else:
            current = current + "\n\n" + para if current else para

    if current.strip():
        sub_chunks.append(current.strip())

    if len(sub_chunks) == 1 and len(sub_chunks[0]) > max_chars:
        sub_chunks = [text[i:i+max_chars] for i in range(0, len(text), max_chars)]

    total = len(sub_chunks)
    return [
        {"id": f"{doc_id} [part {i+1}/{total}]", "text": sc}
        for i, sc in enumerate(sub_chunks)
    ]


print("Utility functions ready")


## Bloc 6 — Per-Chunk Pipeline (Pass 1 + 2 + 3)

In [ ]:
def extract_chunk_pass123(chunk: dict, verbose: bool = True) -> list:
    """
    Runs Pass 1, 2, 3 on a single document chunk.
    Returns a list of translated entities (English).

    New: if Pass 2 times out even after retries, the chunk is automatically
    re-split at half size and each sub-chunk is retried independently.
    """
    doc_id = chunk["id"]
    text   = chunk["text"]

    if verbose:
        print(f"\n   [{doc_id}] ({len(text):,} chars)")

    # -- PASS 1 : Detect which entity types are present ----------------
    if verbose:
        print(f"      Pass 1 — Label detection...", end=" ", flush=True)

    raw1 = call_llm_with_doc(PROMPT_PASS1, text)
    detected_labels = parse_json(raw1, f"{doc_id} Pass1")

    if not detected_labels or not isinstance(detected_labels, list):
        if verbose:
            print("WARNING: No labels detected, chunk skipped")
        return []

    detected_labels = [l for l in detected_labels if l in NODE_LABELS and l != 'Cycle']
    if verbose:
        print(f"{len(detected_labels)} labels: {detected_labels}")

    # -- PASS 2 : Extract entities — with automatic re-split on failure ----
    if verbose:
        print(f"      Pass 2 — Entity extraction...", end=" ", flush=True)

    prompt2 = build_prompt_pass2(detected_labels)

    def _pass2_single(text_fragment: str) -> list | None:
        """Run Pass 2 on a single text fragment. Returns entity list or None."""
        try:
            raw = call_llm_with_doc(prompt2, text_fragment)
            result = parse_json(raw, f"{doc_id} Pass2")
            if result and isinstance(result, list):
                return result
        except Exception as e:
            print(f"FAILED: {e}", end=" ")
        return None

    entities_raw = _pass2_single(text)

    if entities_raw is None:
        # Retry by splitting the chunk in half
        if verbose:
            print(f"\n      Pass 2 RETRY — splitting chunk in half...", end=" ", flush=True)
        half = len(text) // 2
        # cut at nearest paragraph boundary
        cut = text.rfind("\n\n", 0, half) or half
        sub_texts = [text[:cut].strip(), text[cut:].strip()]
        entities_raw = []
        for sub in sub_texts:
            if sub:
                result = _pass2_single(sub)
                if result:
                    entities_raw.extend(result)
        if not entities_raw:
            if verbose:
                print("WARNING: No entities extracted even after re-split")
            return []

    if verbose:
        print(f"{len(entities_raw)} entities (original language)")

    # -- PASS 3 : Translate entity properties to English ---------------
    if verbose:
        print(f"      Pass 3 — Translation to English...", end=" ", flush=True)

    prompt3 = build_prompt_pass3_translate(entities_raw)
    try:
        raw3 = call_llm_no_doc(prompt3)
        entities_translated = parse_json(raw3, f"{doc_id} Pass3")
    except Exception as e:
        print(f"Translation failed ({e}), using original as fallback")
        return entities_raw

    if not entities_translated or not isinstance(entities_translated, list):
        if verbose:
            print("WARNING: Translation failed, using original entities as fallback")
        return entities_raw

    if verbose:
        print(f"{len(entities_translated)} entities translated")

    return entities_translated


print("Per-chunk pipeline (Pass 1+2+3) ready")


## Bloc 7 — Full File Extraction + Global Pass 4

In [ ]:
def run_pass4_batched(all_entities: list, max_batch_size: int = 30) -> list:
    """
    Runs Pass 4 (relationship linking) using affinity-based batching.
    Fix: parameter is max_batch_size (was incorrectly called as batch_size= in v6).
    """
    REL_AFFINITY = [
        ('for',            'Agreement',       'Supplier'),
        ('Use',            'Company',         'Agreement'),
        ('HAS',            'Company',         'Document'),
        ('HAS',            'Company',         'Clients'),
        ('HAS',            'Company',         'governance_body'),
        ('HAS',            'Supplier',        'Company'),
        ('has_requirement','Company',         'Clause'),
        ('apply_to',       'Supplier',        'Clause'),
        ('supervise',      'governance_body', 'Roles'),
        ('approves',       'governance_body', 'Claim'),
        ('Concerns',       'Claim',           'Asset'),
        ('Asserted_by',    'Claim',           'Roles'),
        ('operates',       'Roles',           'Technology'),
        ('EXECUTES',       'Roles',           'Control'),
        ('protects',       'Algorithm',       'Asset'),
        ('used_by',        'Protocol',        'Algorithm'),
        ('mitigates',      'Protocol',        'Risk'),
        ('targeting',      'Risk',            'Asset'),
        ('required_by',    'Framework',       'Control'),
        ('IMPLEMENTED_BY', 'Control',         'Technology'),
        ('Has_access_to',  'Technology',      'Asset'),
    ]

    by_label = {}
    for e in all_entities:
        by_label.setdefault(e['label'], []).append(e)

    def make_sub_batches(from_entities, to_entities, name, max_size):
        combined = from_entities + to_entities
        if len(combined) <= max_size:
            return [(name, combined)]
        n_to = len(to_entities)
        from_slots = max(1, max_size - n_to)
        if from_slots >= len(from_entities):
            from_slots = max_size // 2
            to_slots   = max_size - from_slots
            result = []
            for i in range(0, len(from_entities), from_slots):
                for j in range(0, len(to_entities), to_slots):
                    sub = from_entities[i:i+from_slots] + to_entities[j:j+to_slots]
                    result.append((f"{name}[{i//from_slots},{j//to_slots}]", sub))
            return result
        else:
            result = []
            for i in range(0, len(from_entities), from_slots):
                sub = from_entities[i:i+from_slots] + to_entities
                result.append((f"{name}[{i//from_slots}]", sub))
            return result

    processed_pairs = set()
    all_batches = []
    for rel_type, from_label, to_label in REL_AFFINITY:
        pair = tuple(sorted([from_label, to_label]))
        if pair not in processed_pairs:
            processed_pairs.add(pair)
            from_ents = by_label.get(from_label, [])
            to_ents   = by_label.get(to_label, []) if to_label != from_label else []
            if from_ents or to_ents:
                name = f"{from_label}->{to_label}"
                sub_batches = make_sub_batches(from_ents, to_ents, name, max_batch_size)
                all_batches.extend(sub_batches)

    total = len(all_batches)
    print(f"   Pass 4 — {total} batches (affinity + sub-chunking at max {max_batch_size} entities)")

    all_relationships = []
    seen_relations = set()

    for i, (batch_name, batch_entities) in enumerate(all_batches, 1):
        print(f"      [{i}/{total}] {batch_name} ({len(batch_entities)} entities)...", end=" ", flush=True)
        try:
            prompt4 = build_prompt_pass4_relations(batch_entities)
            # Fix: was call_llm_no_doc_with_retry(...) — now use call_llm_no_doc with built-in retry
            raw4 = call_llm_no_doc(prompt4)
            relationships = parse_json(raw4, f"Pass4 {batch_name}")

            if not relationships or not isinstance(relationships, list):
                print("no relationships")
                continue

            added = 0
            for rel in relationships:
                key = (
                    rel.get("from_label"),
                    str(rel.get("from_key")),
                    rel.get("rel_type"),
                    rel.get("to_label"),
                    str(rel.get("to_key")),
                )
                if key not in seen_relations:
                    seen_relations.add(key)
                    all_relationships.append(rel)
                    added += 1
            print(f"{added} relationships")

        except Exception as e:
            print(f"FAILED after retries: {e}")

    return all_relationships


def extract_file_4pass(
    filepath: str,
    min_chunk_length: int = 200,
    max_chunk_chars: int = 2500,   # reduced from 3000
    pass4_batch_size: int = 50,
) -> dict:
    """
    Loads a file, splits by document marker, runs Pass 1+2+3 per chunk,
    merges entities, then runs Pass 4 in batches globally.
    """
    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()

    all_chunks = split_by_document(text)
    chunks = [c for c in all_chunks if len(c["text"]) >= min_chunk_length]

    final_chunks = []
    for c in chunks:
        sub = split_long_chunk(c, max_chars=max_chunk_chars)
        if len(sub) > 1:
            print(f"   Splitting long chunk '{c['id']}' ({len(c['text']):,} chars) into {len(sub)} parts")
        final_chunks.extend(sub)
    chunks = final_chunks
    print(f"   {len(all_chunks)} chunks found, {len(chunks)} after filtering + splitting")

    # ── PASS 1 + 2 + 3 : per chunk ────────────────────────────────
    all_entities = []
    seen_entities = set()

    for chunk in chunks:
        try:
            entities = extract_chunk_pass123(chunk, verbose=True)
            for entity in entities:
                key = (entity["label"], str(entity.get("properties", {})))
                if key not in seen_entities:
                    seen_entities.add(key)
                    all_entities.append(entity)
        except Exception as e:
            print(f"      WARNING: Error on {chunk['id']} : {e}")

    print(f"\n   Pass 1-3 complete: {len(all_entities)} unique entities (translated, English)")

    # ── PASS 4 : batched relationship linking ─────────────────────
    # Fix: was batch_size= (wrong kwarg), now correctly max_batch_size=
    relationships = run_pass4_batched(all_entities, max_batch_size=pass4_batch_size)

    print(f"   DONE: {len(all_entities)} entities, {len(relationships)} relationships")
    return {"entities": all_entities, "relationships": relationships}


print("Full extraction function ready")


## Bloc 8 — Run Extraction

In [ ]:
DOCUMENTS = [
    "/home/maroua/Desktop/cassiope/SecuraOps High.txt",
    # "/home/maroua/Desktop/cassiope/SafeLink Contradictory.txt",
]

raw_results = {}

for filepath in DOCUMENTS:
    nom = filepath.split("/")[-1]
    print(f"\n{'='*60}")
    print(f"Extraction: {nom}")
    print(f"{'='*60}")
    try:
        result = extract_file_4pass(filepath)
        raw_results[filepath] = result
    except Exception as e:
        print(f"ERROR: {e}")
        raw_results[filepath] = {"entities": [], "relationships": []}


## Bloc 9 — Results Visualization

In [ ]:
from collections import Counter

for filepath, result in raw_results.items():
    nom = filepath.split("/")[-1]
    print(f"\n{nom}")
    print(f"   Entities      : {len(result['entities'])}")
    print(f"   Relationships : {len(result['relationships'])}")
    node_counts = Counter(e['label'] for e in result['entities'])
    print("   Entity types:")
    for label, count in sorted(node_counts.items()):
        print(f"      {label:<20} : {count}")

    print("\n   Sample entities (first 5, should be in English):")
    for e in result['entities'][:5]:
        print(f"      [{e['label']}] {e['properties']}")


## Bloc 10 — JSON Save + Auto Post-Processing

In [ ]:
from copy import deepcopy

KEY_FIELDS = {
    'Company': 'name', 'Supplier': 'name', 'Agreement': 'name',
    'Document': 'reference', 'Clients': 'name', 'Clause': 'name',
    'governance_body': 'name', 'Claim': 'claim_type', 'Roles': 'title',
    'Asset': 'type', 'Algorithm': 'name', 'Protocol': 'name',
    'Risk': 'description', 'Framework': 'name', 'Control': 'name',
    'Technology': 'name'
}

def merge_props(a, b):
    m = deepcopy(a)
    for k, v in b.items():
        if v and (not m.get(k) or len(str(v)) > len(str(m.get(k,'')))):
            m[k] = v
    return m

def postprocess(result: dict) -> dict:
    entities = result['entities']
    relationships = result['relationships']
    print(f"  Before: {len(entities)} entities, {len(relationships)} relationships")

    seen, deduped = {}, []
    for e in entities:
        kf = KEY_FIELDS.get(e['label'])
        kv = e['properties'].get(kf, '').strip().lower() if kf else ''
        dk = (e['label'], kv) if kv else None
        if dk and dk in seen:
            deduped[seen[dk]]['properties'] = merge_props(deduped[seen[dk]]['properties'], e['properties'])
        else:
            deduped.append(deepcopy(e))
            if dk: seen[dk] = len(deduped) - 1
    print(f"  Dedup : {len(entities)} → {len(deduped)} entities")

    valid_entities = []
    for e in deduped:
        kf = KEY_FIELDS.get(e['label'])
        if kf and not e['properties'].get(kf, '').strip():
            print(f"  Drop  : [{e['label']}] {e['properties']}")
        else:
            valid_entities.append(e)

    index = set()
    for e in valid_entities:
        index.add((e['label'], frozenset(e['properties'].items())))

    def matches(label, key_dict):
        return any(l == label and all(dict(p).get(k) == v for k, v in key_dict.items())
                   for l, p in index)

    valid_rels = [r for r in relationships
                  if matches(r['from_label'], r['from_key']) and matches(r['to_label'], r['to_key'])]
    print(f"  Rels  : {len(relationships)} → {len(valid_rels)} (removed {len(relationships)-len(valid_rels)} broken)")
    print(f"  After : {len(valid_entities)} entities, {len(valid_rels)} relationships")
    return {'entities': valid_entities, 'relationships': valid_rels}


for filepath, result in raw_results.items():
    nom = filepath.split('/')[-1].replace('.txt', '')

    raw_file = f"{nom}_graph_4pass.json"
    with open(raw_file, 'w', encoding='utf-8') as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
    print(f"Raw saved   : {raw_file}")

    print(f"Post-processing {nom}...")
    clean = postprocess(result)
    clean_file = f"{nom}_graph_4pass_clean.json"
    with open(clean_file, 'w', encoding='utf-8') as f:
        json.dump(clean, f, indent=2, ensure_ascii=False)
    print(f"Clean saved : {clean_file}\n")


## Bloc 11 — Cypher Query Generation

In [ ]:
def props_str(props: dict) -> str:
    parts = [f"{k}: {json.dumps(v)}" for k, v in props.items()]
    return "{" + ", ".join(parts) + "}"


def json_to_cypher(extracted: dict, company_label: str = None) -> list:
    statements = []
    for entity in extracted.get("entities", []):
        label = entity["label"]
        props = entity.get("properties", {})
        if props:
            if company_label:
                statements.append(f"MERGE (n:{label}:{company_label} {props_str(props)});")
            else:
                statements.append(f"MERGE (n:{label} {props_str(props)});")
    for rel in extracted.get("relationships", []):
        fl = rel["from_label"]
        fk = props_str(rel["from_key"])
        rt = rel["rel_type"]
        tl = rel["to_label"]
        tk = props_str(rel["to_key"])
        if company_label:
            statements.append(
                f"MATCH (a:{fl}:{company_label} {fk}), (b:{tl}:{company_label} {tk}) MERGE (a)-[:{rt}]->(b);"
            )
        else:
            statements.append(
                f"MATCH (a:{fl} {fk}), (b:{tl} {tk}) MERGE (a)-[:{rt}]->(b);"
            )
    return statements


company_map = {
    "SecuraOps High": "SecuraOps",
    "SafeLink Contradictory": "SafeLink",
}

cypher_by_file = {}
for filepath, result in raw_results.items():
    nom = filepath.split("/")[-1].replace(".txt", "")
    company = company_map.get(nom, nom)
    stmts = json_to_cypher(result, company_label=company)
    cypher_by_file[nom] = stmts
    print(f"OK {nom} : {len(stmts)} Cypher statements")


## Bloc 12 — Send to Neo4j

In [ ]:
from neo4j import GraphDatabase

uri      = os.getenv("NEO4J_URI",      "bolt://localhost:7687")
user     = os.getenv("NEO4J_USER",     "neo4j")
password = os.getenv("NEO4J_PASSWORD", "")

driver = GraphDatabase.driver(uri, auth=(user, password))

with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")
print("Database cleared")

for nom, stmts in cypher_by_file.items():
    print(f"\nSending {nom} to Neo4j...")
    errors = 0
    with driver.session() as session:
        for stmt in stmts:
            try:
                session.run(stmt)
            except Exception as e:
                errors += 1
    print(f"   {len(stmts) - errors} statements OK")
    if errors:
        print(f"   {errors} errors")

driver.close()
print("\nDone!")
